# 5_export_tableau_dm.ipynb

이 노트북은 **(Early / Full) 모델 SHAP 산출물**을 입력으로 받아 Tableau에서 바로 쓰기 좋은 형태의 **경량 DM**을 생성합니다.

## 생성하는 DM (권장)
### Tab2 (Global 구조)
- `dm_tableau_tab2_full_global_topn.csv`
- `dm_tableau_tab2_early_global_topn.csv`
- `dm_tableau_tab2_full_group_share.csv`
- `dm_tableau_tab2_early_group_share.csv`

### Tab3 (포지션 구조)
- `dm_tableau_tab3_full_group_positions.csv`
- `dm_tableau_tab3_full_position_top_features.csv` (포지션별 Top feature 표; Hide 토글 적용용)

## 핵심 처리
- `shap_share` (Global) 및 `group_share` (Group) 컬럼 생성
- Tableau에서 Hide(예: Hide Gold)를 유연하게 하기 위해, **feature_name/metric_group/feature_group 컬럼은 유지**
- Tableau에서 key/필터로 쓰기 어려운 컬럼은 최소화(슬림화)


In [3]:
import os
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)


In [4]:
# =========================
# CONFIG
# =========================

# 프로젝트 경로

# 입력 폴더
IN_EARLY_DIR = r"../../data/Riot_API/feature/datamart/early"
IN_FULL_DIR  = r"../../data/Riot_API/feature/datamart/full"

# 출력 폴더
OUT_TAB2_DIR = r"../../data/Riot_API/feature/tableau/TAB2"
OUT_TAB3_DIR = r"../../data/Riot_API/feature/tableau/TAB3"
os.makedirs(OUT_TAB2_DIR, exist_ok=True)
os.makedirs(OUT_TAB3_DIR, exist_ok=True)

# Top-N 설정
TOPN_FULL_GLOBAL  = 25
TOPN_EARLY_GLOBAL = 12  # early feature=15 기준 권장(12~15)

TOPN_POS_FEATURES = 10  # Tab3 '포지션별 Top features table'

print("IN_EARLY_DIR:", IN_EARLY_DIR)
print("IN_FULL_DIR :", IN_FULL_DIR)
print("OUT_TAB2_DIR:", OUT_TAB2_DIR)
print("OUT_TAB3_DIR:", OUT_TAB3_DIR)


IN_EARLY_DIR: ../../data/Riot_API/feature/datamart/early
IN_FULL_DIR : ../../data/Riot_API/feature/datamart/full
OUT_TAB2_DIR: ../../data/Riot_API/feature/tableau/TAB2
OUT_TAB3_DIR: ../../data/Riot_API/feature/tableau/TAB3


In [5]:
# =========================
# INPUT FILES
# =========================
PATH_FULL_GLOBAL = os.path.join(IN_FULL_DIR, "dm_match_team_shap_global_full.csv")
PATH_FULL_GROUP  = os.path.join(IN_FULL_DIR, "dm_match_team_shap_group_full.csv")

PATH_EARLY_GLOBAL = os.path.join(IN_EARLY_DIR, "dm_match_team_shap_global_early.csv")
PATH_EARLY_GROUP  = os.path.join(IN_EARLY_DIR, "dm_match_team_shap_group_early.csv")

# (선택) local은 Tableau Tab4가 아니라 Streamlit에서 주로 사용
PATH_FULL_LOCAL = os.path.join(IN_FULL_DIR, "dm_match_team_shap_local_full_with_meta.csv")
PATH_EARLY_LOCAL = os.path.join(IN_EARLY_DIR, "dm_match_team_shap_local_early_with_meta.csv")

for p in [PATH_FULL_GLOBAL, PATH_FULL_GROUP, PATH_EARLY_GLOBAL, PATH_EARLY_GROUP]:
    print(os.path.exists(p), p)


True ../../data/Riot_API/feature/datamart/full\dm_match_team_shap_global_full.csv
True ../../data/Riot_API/feature/datamart/full\dm_match_team_shap_group_full.csv
True ../../data/Riot_API/feature/datamart/early\dm_match_team_shap_global_early.csv
True ../../data/Riot_API/feature/datamart/early\dm_match_team_shap_group_early.csv


In [6]:
# =========================
# LOAD
# =========================
full_global = pd.read_csv(PATH_FULL_GLOBAL)
full_group  = pd.read_csv(PATH_FULL_GROUP)

early_global = pd.read_csv(PATH_EARLY_GLOBAL)
early_group  = pd.read_csv(PATH_EARLY_GROUP)

print("full_global:", full_global.shape)
print("full_group :", full_group.shape)
print("early_global:", early_global.shape)
print("early_group :", early_group.shape)

display(full_global.head(3))
display(full_group.head(3))


full_global: (166, 15)
full_group : (14, 14)
early_global: (30, 14)
early_group : (8, 14)


,model_name,variant,split,seed,n_estimators,base_logit,base_prob,feature_name,feature_group,metric_group,mean_abs_shap_logit,mean_signed_shap_logit,mean_abs_delta_p,mean_signed_delta_p,rank
0,lightgbm,deleaked83,matchid_random_80_20,42,800,-0.008508,0.497873,pos_SUP__gold_earned__diff,SUP,ECON,1.095483,-0.003524,0.228270,0.001154,1
1,lightgbm,deleaked83,matchid_random_80_20,42,800,-0.008508,0.497873,pos_MID__gold_earned__diff,MID,ECON,0.976919,0.001408,0.211998,0.001631,2
2,lightgbm,deleaked83,matchid_random_80_20,42,800,-0.008508,0.497873,pos_TOP__gold_earned__diff,TOP,ECON,0.972125,-0.004737,0.208335,0.000903,3


,model_name,variant,split,seed,n_estimators,base_logit,base_prob,group,n_features_in_group,mean_abs_shap_logit,mean_signed_shap_logit,mean_abs_delta_p,mean_signed_delta_p,rank
0,lightgbm,deleaked83,matchid_random_80_20,42,800,-0.008508,0.497873,TEAM_DIFF,13,1.707158,0.008713,0.323942,0.001678,1
1,lightgbm,deleaked83,matchid_random_80_20,42,800,-0.008508,0.497873,SUP,13,1.158671,-0.000575,0.237605,0.001975,2
2,lightgbm,deleaked83,matchid_random_80_20,42,800,-0.008508,0.497873,TOP,13,1.107246,-0.005229,0.229742,0.001088,3


In [7]:
# =========================
# HELPERS
# =========================
def add_share(df: pd.DataFrame, value_col: str, by_cols: list, out_col: str) -> pd.DataFrame:
    """value_col / sum(value_col) within by_cols"""
    tmp = df.copy()
    denom = tmp.groupby(by_cols)[value_col].transform("sum")
    tmp[out_col] = np.where(denom > 0, tmp[value_col] / denom, np.nan)
    return tmp

def topn_by_model(df: pd.DataFrame, n: int, sort_col: str = "mean_abs_shap_logit") -> pd.DataFrame:
    out = (df.sort_values(["model_name", sort_col], ascending=[True, False])
             .groupby(["model_name","variant","split","seed"], as_index=False, group_keys=False)
             .head(n)
             .reset_index(drop=True))
    return out

def slim_global(df: pd.DataFrame) -> pd.DataFrame:
    keep = [c for c in [
        "model_name","variant","split","seed","n_estimators",
        "base_logit","base_prob",
        "rank","feature_name","feature_group","metric_group",
        "mean_abs_shap_logit","mean_signed_shap_logit",
        "mean_abs_delta_p","mean_signed_delta_p",
        "shap_share"
    ] if c in df.columns]
    return df[keep].copy()

def slim_group(df: pd.DataFrame) -> pd.DataFrame:
    keep = [c for c in [
        "model_name","variant","split","seed","n_estimators",
        "group","n_features_in_group",
        "mean_abs_shap_logit","mean_signed_shap_logit",
        "mean_abs_delta_p","mean_signed_delta_p",
        "group_share","rank"
    ] if c in df.columns]
    return df[keep].copy()


In [8]:
# =========================
# Tab2: Full Global TopN + shap_share
# =========================
full_global2 = add_share(full_global, "mean_abs_shap_logit",
                         by_cols=["model_name","variant","split","seed"],
                         out_col="shap_share")
full_topn = topn_by_model(full_global2, TOPN_FULL_GLOBAL, sort_col="mean_abs_shap_logit")
full_topn = slim_global(full_topn)

OUT_TAB2_FULL_TOPN = os.path.join(OUT_TAB2_DIR, "dm_tableau_tab2_full_global_topn.csv")
full_topn.to_csv(OUT_TAB2_FULL_TOPN, index=False)
print("Saved:", OUT_TAB2_FULL_TOPN, full_topn.shape)

display(full_topn.head(10))


Saved: ../../data/Riot_API/feature/tableau/TAB2\dm_tableau_tab2_full_global_topn.csv (50, 16)


,model_name,variant,split,seed,n_estimators,base_logit,base_prob,rank,feature_name,feature_group,metric_group,mean_abs_shap_logit,mean_signed_shap_logit,mean_abs_delta_p,mean_signed_delta_p,shap_share
0,lightgbm,deleaked83,matchid_random_80_20,42,800,-0.008508,0.497873,1,pos_SUP__gold_earned__diff,SUP,ECON,1.095483,-0.003524,0.228270,0.001154,0.125339
1,lightgbm,deleaked83,matchid_random_80_20,42,800,-0.008508,0.497873,2,pos_MID__gold_earned__diff,MID,ECON,0.976919,0.001408,0.211998,0.001631,0.111773
2,lightgbm,deleaked83,matchid_random_80_20,42,800,-0.008508,0.497873,3,pos_TOP__gold_earned__diff,TOP,ECON,0.972125,-0.004737,0.208335,0.000903,0.111225
3,lightgbm,deleaked83,matchid_random_80_20,42,800,-0.008508,0.497873,4,pos_JNG__gold_earned__diff,JNG,ECON,0.965457,-0.001065,0.207842,0.001325,0.110462
4,lightgbm,deleaked83,matchid_random_80_20,42,800,-0.008508,0.497873,5,pos_BOT__gold_earned__diff,BOT,ECON,0.707279,-0.004184,0.163728,-0.000712,0.080923
5,lightgbm,deleaked83,matchid_random_80_20,42,800,-0.008508,0.497873,6,kills_diff,TEAM_DIFF,COMBAT,0.622918,0.001261,0.149366,0.000478,0.071271
6,lightgbm,deleaked83,matchid_random_80_20,42,800,-0.008508,0.497873,7,deaths_diff,TEAM_DIFF,COMBAT,0.489977,0.002164,0.119197,0.000617,0.056060
7,lightgbm,deleaked83,matchid_random_80_20,42,800,-0.008508,0.497873,8,dragon_kills_diff,TEAM_DIFF,OBJECTIVE,0.411963,-0.003035,0.099962,-0.000591,0.047134
8,lightgbm,deleaked83,matchid_random_80_20,42,800,-0.008508,0.497873,9,gold_spent_diff,TEAM_DIFF,ECON,0.303429,0.008453,0.074893,0.002098,0.034717
9,lightgbm,deleaked83,matchid_random_80_20,42,800,-0.008508,0.497873,10,baron_kills_diff,TEAM_DIFF,OBJECTIVE,0.278371,-0.000148,0.068638,0.000032,0.031850


In [9]:
# =========================
# Tab2: Early Global TopN + shap_share
# =========================
early_global2 = add_share(early_global, "mean_abs_shap_logit",
                          by_cols=["model_name","variant","split","seed"],
                          out_col="shap_share")

n_early_features = early_global2.groupby(["model_name","variant","split","seed"]).size().min()
TOPN_EARLY = int(min(TOPN_EARLY_GLOBAL, n_early_features))
print("Early TOPN used:", TOPN_EARLY)

early_topn = topn_by_model(early_global2, TOPN_EARLY, sort_col="mean_abs_shap_logit")
early_topn = slim_global(early_topn)

OUT_TAB2_EARLY_TOPN = os.path.join(OUT_TAB2_DIR, "dm_tableau_tab2_early_global_topn.csv")
early_topn.to_csv(OUT_TAB2_EARLY_TOPN, index=False)
print("Saved:", OUT_TAB2_EARLY_TOPN, early_topn.shape)

display(early_topn.head(10))


Early TOPN used: 12
Saved: ../../data/Riot_API/feature/tableau/TAB2\dm_tableau_tab2_early_global_topn.csv (24, 14)


,model_name,variant,split,seed,n_estimators,base_logit,base_prob,rank,feature_name,mean_abs_shap_logit,mean_signed_shap_logit,mean_abs_delta_p,mean_signed_delta_p,shap_share
0,lightgbm,early10_all_features,matchid_random_80_20,42,350,0.000015,0.500004,1,gold_diff_10,0.323747,0.001464,0.079283,0.000337,0.260993
1,lightgbm,early10_all_features,matchid_random_80_20,42,350,0.000015,0.500004,2,evt_elite_monster_kill_10_diff_10,0.196363,-0.000594,0.048631,-0.000148,0.158300
2,lightgbm,early10_all_features,matchid_random_80_20,42,350,0.000015,0.500004,3,evt_champion_kill_10_diff_10,0.182314,-0.000761,0.045215,-0.000185,0.146975
3,lightgbm,early10_all_features,matchid_random_80_20,42,350,0.000015,0.500004,4,cs_diff_10,0.160441,0.000480,0.039820,0.000122,0.129341
4,lightgbm,early10_all_features,matchid_random_80_20,42,350,0.000015,0.500004,5,evt_turret_plate_destroyed_10_diff_10,0.093281,-0.000376,0.023275,-0.000093,0.075200
5,lightgbm,early10_all_features,matchid_random_80_20,42,350,0.000015,0.500004,6,elite_dragon_10_diff_10,0.090603,0.000070,0.022610,0.000017,0.073041
6,lightgbm,early10_all_features,matchid_random_80_20,42,350,0.000015,0.500004,7,xp_diff_10,0.089260,0.000209,0.022281,0.000051,0.071958
7,lightgbm,early10_all_features,matchid_random_80_20,42,350,0.000015,0.500004,8,enemy_cc_time_diff_10,0.032710,-0.000008,0.008176,-0.000002,0.026369
8,lightgbm,early10_all_features,matchid_random_80_20,42,350,0.000015,0.500004,9,dmg_to_champ_diff_10,0.028283,-0.000852,0.007069,-0.000213,0.022800
9,lightgbm,early10_all_features,matchid_random_80_20,42,350,0.000015,0.500004,10,elite_horde_10_diff_10,0.015048,0.000456,0.003762,0.000114,0.012131


In [10]:
# =========================
# Tab2: Group share (Full / Early)
# =========================
full_group2 = add_share(full_group, "mean_abs_shap_logit",
                        by_cols=["model_name","variant","split","seed"],
                        out_col="group_share")
full_group2["rank"] = full_group2.groupby(["model_name","variant","split","seed"])["mean_abs_shap_logit"]\
                                 .rank(ascending=False, method="dense").astype(int)

early_group2 = add_share(early_group, "mean_abs_shap_logit",
                         by_cols=["model_name","variant","split","seed"],
                         out_col="group_share")
early_group2["rank"] = early_group2.groupby(["model_name","variant","split","seed"])["mean_abs_shap_logit"]\
                                   .rank(ascending=False, method="dense").astype(int)

full_group_out = slim_group(full_group2)
early_group_out = slim_group(early_group2)

OUT_TAB2_FULL_GROUP = os.path.join(OUT_TAB2_DIR, "dm_tableau_tab2_full_group_share.csv")
OUT_TAB2_EARLY_GROUP = os.path.join(OUT_TAB2_DIR, "dm_tableau_tab2_early_group_share.csv")

full_group_out.to_csv(OUT_TAB2_FULL_GROUP, index=False)
early_group_out.to_csv(OUT_TAB2_EARLY_GROUP, index=False)

print("Saved:", OUT_TAB2_FULL_GROUP, full_group_out.shape)
print("Saved:", OUT_TAB2_EARLY_GROUP, early_group_out.shape)

display(full_group_out.sort_values(["model_name","rank"]).head(20))
display(early_group_out.sort_values(["model_name","rank"]).head(20))


Saved: ../../data/Riot_API/feature/tableau/TAB2\dm_tableau_tab2_full_group_share.csv (14, 13)
Saved: ../../data/Riot_API/feature/tableau/TAB2\dm_tableau_tab2_early_group_share.csv (8, 13)


,model_name,variant,split,seed,n_estimators,group,n_features_in_group,mean_abs_shap_logit,mean_signed_shap_logit,mean_abs_delta_p,mean_signed_delta_p,group_share,rank
0,lightgbm,deleaked83,matchid_random_80_20,42,800,TEAM_DIFF,13,1.707158,0.008713,0.323942,0.001678,0.249086,1
1,lightgbm,deleaked83,matchid_random_80_20,42,800,SUP,13,1.158671,-0.000575,0.237605,0.001975,0.169058,2
2,lightgbm,deleaked83,matchid_random_80_20,42,800,TOP,13,1.107246,-0.005229,0.229742,0.001088,0.161555,3
3,lightgbm,deleaked83,matchid_random_80_20,42,800,JNG,13,1.042253,-0.000244,0.220956,0.001566,0.152072,4
4,lightgbm,deleaked83,matchid_random_80_20,42,800,MID,13,1.024177,0.002776,0.219400,0.001901,0.149434,5
5,lightgbm,deleaked83,matchid_random_80_20,42,800,BOT,13,0.718610,-0.003793,0.166357,-0.000704,0.104850,6
6,lightgbm,deleaked83,matchid_random_80_20,42,800,STRUCT,5,0.095579,0.000309,0.023806,0.000011,0.013946,7
7,xgboost,deleaked83,matchid_random_80_20,42,800,TEAM_DIFF,13,1.605205,0.001994,0.312490,0.000058,0.248606,1
8,xgboost,deleaked83,matchid_random_80_20,42,800,SUP,13,1.080342,0.000362,0.226133,0.001280,0.167318,2
9,xgboost,deleaked83,matchid_random_80_20,42,800,TOP,13,1.035172,0.004979,0.219478,0.002275,0.160322,3


,model_name,variant,split,seed,n_estimators,group,n_features_in_group,mean_abs_shap_logit,mean_signed_shap_logit,mean_abs_delta_p,mean_signed_delta_p,group_share,rank
0,lightgbm,early10_all_features,matchid_random_80_20,42,350,EARLY_OTHER,5,0.497546,0.001810,0.117743,0.000375,0.508795,1
1,lightgbm,early10_all_features,matchid_random_80_20,42,350,EARLY_OBJECTIVE,4,0.257706,-0.000444,0.063524,-0.000110,0.263532,2
2,lightgbm,early10_all_features,matchid_random_80_20,42,350,EARLY_COMBAT,4,0.206856,-0.001250,0.051182,-0.000300,0.211532,3
3,lightgbm,early10_all_features,matchid_random_80_20,42,350,EARLY_VISION,2,0.015784,0.000028,0.003945,0.000007,0.016141,4
4,xgboost,early10_all_features,matchid_random_80_20,42,350,EARLY_OTHER,5,0.493418,-0.001705,0.116811,-0.000373,0.501790,1
5,xgboost,early10_all_features,matchid_random_80_20,42,350,EARLY_OBJECTIVE,4,0.256996,0.000443,0.063365,0.000103,0.261357,2
6,xgboost,early10_all_features,matchid_random_80_20,42,350,EARLY_COMBAT,4,0.213856,0.001222,0.052908,0.000304,0.217484,3
7,xgboost,early10_all_features,matchid_random_80_20,42,350,EARLY_VISION,2,0.019047,0.000082,0.004760,0.000020,0.019370,4


In [11]:
# =========================
# Tab3: Full Group Positions + shares
# =========================
POS = ["TOP","JNG","MID","BOT","SUP"]

full_group_pos = full_group2[full_group2["group"].isin(POS)].copy()

# share within positions only (for bar chart that sums to 100% across positions)
full_group_pos = add_share(full_group_pos, "mean_abs_shap_logit",
                           by_cols=["model_name","variant","split","seed"],
                           out_col="pos_share")

keep = [c for c in [
    "model_name","variant","split","seed","n_estimators",
    "group","n_features_in_group",
    "mean_abs_shap_logit","mean_signed_shap_logit",
    "mean_abs_delta_p","mean_signed_delta_p",
    "group_share","pos_share","rank"
] if c in full_group_pos.columns]
full_group_pos_out = full_group_pos[keep].copy()

OUT_TAB3_FULL_POS = os.path.join(OUT_TAB3_DIR, "dm_tableau_tab3_full_group_positions.csv")
full_group_pos_out.to_csv(OUT_TAB3_FULL_POS, index=False)
print("Saved:", OUT_TAB3_FULL_POS, full_group_pos_out.shape)

display(full_group_pos_out.sort_values(["model_name","rank"]).head(20))


Saved: ../../data/Riot_API/feature/tableau/TAB3\dm_tableau_tab3_full_group_positions.csv (10, 14)


,model_name,variant,split,seed,n_estimators,group,n_features_in_group,mean_abs_shap_logit,mean_signed_shap_logit,mean_abs_delta_p,mean_signed_delta_p,group_share,pos_share,rank
1,lightgbm,deleaked83,matchid_random_80_20,42,800,SUP,13,1.158671,-0.000575,0.237605,0.001975,0.169058,0.229396,2
2,lightgbm,deleaked83,matchid_random_80_20,42,800,TOP,13,1.107246,-0.005229,0.229742,0.001088,0.161555,0.219215,3
3,lightgbm,deleaked83,matchid_random_80_20,42,800,JNG,13,1.042253,-0.000244,0.220956,0.001566,0.152072,0.206348,4
4,lightgbm,deleaked83,matchid_random_80_20,42,800,MID,13,1.024177,0.002776,0.219400,0.001901,0.149434,0.202769,5
5,lightgbm,deleaked83,matchid_random_80_20,42,800,BOT,13,0.718610,-0.003793,0.166357,-0.000704,0.104850,0.142272,6
8,xgboost,deleaked83,matchid_random_80_20,42,800,SUP,13,1.080342,0.000362,0.226133,0.001280,0.167318,0.228648,2
9,xgboost,deleaked83,matchid_random_80_20,42,800,TOP,13,1.035172,0.004979,0.219478,0.002275,0.160322,0.219088,3
10,xgboost,deleaked83,matchid_random_80_20,42,800,MID,13,0.950178,0.004270,0.206801,0.001469,0.147159,0.201099,4
11,xgboost,deleaked83,matchid_random_80_20,42,800,JNG,13,0.927625,0.001078,0.202574,0.000866,0.143666,0.196326,5
12,xgboost,deleaked83,matchid_random_80_20,42,800,BOT,13,0.731606,-0.001894,0.169355,-0.000585,0.113307,0.154840,6


In [12]:
# =========================
# Tab3: Full Position Top Features Table (Hide 토글 적용용)
# =========================
full_global_pos = full_global2[full_global2["feature_group"].isin(POS)].copy()

# position 내 share (포지션 내부에서의 상대적 비중)
full_global_pos = add_share(full_global_pos, "mean_abs_shap_logit",
                            by_cols=["model_name","variant","split","seed","feature_group"],
                            out_col="pos_feature_share")

full_pos_top = (full_global_pos.sort_values(["model_name","feature_group","mean_abs_shap_logit"], ascending=[True, True, False])
                            .groupby(["model_name","variant","split","seed","feature_group"], as_index=False, group_keys=False)
                            .head(TOPN_POS_FEATURES)
                            .reset_index(drop=True))

full_pos_top["rank_in_position"] = full_pos_top.groupby(["model_name","variant","split","seed","feature_group"])["mean_abs_shap_logit"]\
                                        .rank(ascending=False, method="first").astype(int)

keep = [c for c in [
    "model_name","variant","split","seed","n_estimators",
    "feature_group","feature_name","metric_group",
    "mean_abs_shap_logit","mean_signed_shap_logit",
    "mean_abs_delta_p","mean_signed_delta_p",
    "shap_share","pos_feature_share","rank_in_position"
] if c in full_pos_top.columns]

full_pos_top_out = full_pos_top[keep].copy()

OUT_TAB3_POS_TOPFEAT = os.path.join(OUT_TAB3_DIR, "dm_tableau_tab3_full_position_top_features.csv")
full_pos_top_out.to_csv(OUT_TAB3_POS_TOPFEAT, index=False)
print("Saved:", OUT_TAB3_POS_TOPFEAT, full_pos_top_out.shape)

display(full_pos_top_out.head(20))


Saved: ../../data/Riot_API/feature/tableau/TAB3\dm_tableau_tab3_full_position_top_features.csv (100, 15)


,model_name,variant,split,seed,n_estimators,feature_group,feature_name,metric_group,mean_abs_shap_logit,mean_signed_shap_logit,mean_abs_delta_p,mean_signed_delta_p,shap_share,pos_feature_share,rank_in_position
0,lightgbm,deleaked83,matchid_random_80_20,42,800,BOT,pos_BOT__gold_earned__diff,ECON,0.707279,-0.004184,0.163728,-7.118791e-04,0.080923,0.777927,1
1,lightgbm,deleaked83,matchid_random_80_20,42,800,BOT,pos_BOT__kills__diff,COMBAT,0.077735,0.000935,0.019411,2.344292e-04,0.008894,0.085500,2
2,lightgbm,deleaked83,matchid_random_80_20,42,800,BOT,pos_BOT__ch_takedowns__diff,COMBAT,0.050825,0.000156,0.012702,4.046135e-05,0.005815,0.055902,3
3,lightgbm,deleaked83,matchid_random_80_20,42,800,BOT,pos_BOT__dmg_mitigated__diff,DAMAGE,0.033466,-0.000021,0.008365,-4.678565e-06,0.003829,0.036809,4
4,lightgbm,deleaked83,matchid_random_80_20,42,800,BOT,pos_BOT__dmg_taken__diff,DAMAGE,0.018846,-0.000646,0.004711,-1.612676e-04,0.002156,0.020729,5
5,lightgbm,deleaked83,matchid_random_80_20,42,800,BOT,pos_BOT__lane_cs__diff,ECON,0.004772,-0.000060,0.001193,-1.494538e-05,0.000546,0.005249,6
6,lightgbm,deleaked83,matchid_random_80_20,42,800,BOT,pos_BOT__dmg_to_champ__diff,DAMAGE,0.004080,0.000003,0.001020,7.054570e-07,0.000467,0.004487,7
7,lightgbm,deleaked83,matchid_random_80_20,42,800,BOT,pos_BOT__vision_score__diff,VISION,0.003274,-0.000101,0.000818,-2.534953e-05,0.000375,0.003601,8
8,lightgbm,deleaked83,matchid_random_80_20,42,800,BOT,pos_BOT__ch_kill_participation__diff,COMBAT,0.003092,0.000277,0.000773,6.920410e-05,0.000354,0.003401,9
9,lightgbm,deleaked83,matchid_random_80_20,42,800,BOT,pos_BOT__deaths__diff,COMBAT,0.002947,-0.000126,0.000737,-3.147615e-05,0.000337,0.003241,10


In [13]:
# =========================
# BASIC VALIDATION (sanity)
# =========================
def check_topn(df, expected_n):
    sizes = df.groupby("model_name").size().to_dict()
    print("rows per model:", sizes, "| expected per model:", expected_n)

print("Tab2 Full TopN:")
check_topn(full_topn, TOPN_FULL_GLOBAL)

print("Tab2 Early TopN:")
check_topn(early_topn, TOPN_EARLY)

print("Tab3 positions rows per model (should be 5):")
print(full_group_pos_out.groupby("model_name").size().to_dict())

print("Tab3 position top features rows per model (should be 5*TOPN_POS_FEATURES):")
print(full_pos_top_out.groupby("model_name").size().to_dict())

print("Done.")


Tab2 Full TopN:
rows per model: {'lightgbm': 25, 'xgboost': 25} | expected per model: 25
Tab2 Early TopN:
rows per model: {'lightgbm': 12, 'xgboost': 12} | expected per model: 12
Tab3 positions rows per model (should be 5):
{'lightgbm': 5, 'xgboost': 5}
Tab3 position top features rows per model (should be 5*TOPN_POS_FEATURES):
{'lightgbm': 50, 'xgboost': 50}
Done.
